# Nazih — Training Set Size (reuses earlier 70-image results)

**Do not execute without a matching CPU environment.** This notebook does not claim new training has occurred. Upload the ZIP package and run these commands in order. Prior 70-image results remain untouched; only new sizes 140, 280, 420 are trained.

In [ ]:
from google.colab import files
import zipfile, pathlib, subprocess, sys, os
uploaded=files.upload()
packages=[p for p in uploaded if p.startswith('Nazih_Training_Set_Size_') and p.endswith('.zip')]
assert len(packages)==1, 'Upload the study ZIP file, not the 500 MB weights archive'
base=pathlib.Path('/content/nazih_sample_size')
base.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(packages[0]) as z:
    assert all(not p.startswith('/') and '..' not in pathlib.PurePosixPath(p).parts for p in z.namelist())
    z.extractall(base)
print('STUDY:',base)

In [ ]:
def cmd(*args):
    subprocess.run([sys.executable, *map(str,args)],cwd=base,check=True)
cmd('scripts/preflight.py')
cmd('scripts/test_pipeline.py')
print('Current Python:',sys.version)

## Environment requirement

The archived models used **CPU, torch 2.15.0+cpu, torchvision 0.29.2+cpu, numpy 2.5.4, Pillow 13.0.0**. Running on GPU or with mismatched packages is not a clean sample-size-only comparison. Set up an environment with those exact versions first. The following cell checks and stops on mismatch rather than silently retraining the existing 70-image baseline.

In [ ]:
sys.path.insert(0,str(base/'scripts'))
from train import test_environment
try:
    print('Reference environment OK:',test_environment())
except Exception as err:
    print('BLOCKED - install/match original CPU environment before training:',err)
    raise

## Official dataset ZIP

This is the **original** KSDD2 archive, different from the 23MB diagnostic ZIP. The code downloads to Colab once if missing, verifies the full archive SHA256, and then selects 140/280/420 original training images. If the official download is inaccessible, obtain the same original ZIP from the source and put it at `/content/KolektorSDD2.zip`.

In [ ]:
import urllib.request
archive=pathlib.Path('/content/KolektorSDD2.zip')
if not archive.exists():
    print('Retrieving original KSDD2 ZIP from official data.vicos.si source...')
    tmp=archive.with_suffix('.zip.part')
    try:
        urllib.request.urlretrieve('https://data.vicos.si/datasets/KSDD/KolektorSDD2.zip',tmp)
        tmp.replace(archive)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise
cmd('scripts/prepare.py','--archive',archive)

## NEW training only — 18 models

CPU training at 448px is resource-intensive. Store the extracted study folder on a persistent drive if your Colab runtime might expire. Output checkpoints are never overwritten; completed conditions are skipped below.

In [ ]:
for size in (140,280,420):
    for seed in (20261028,20261029,20261030):
        for loss in ('ce','poly1'):
            result=base/f'results/size{size}_seed{seed}_{loss}.json'
            weight=base/f'results/size{size}_seed{seed}_{loss}.pt'
            if result.exists() and weight.exists():
                print('PRESERVED:',size,seed,loss);continue
            cmd('scripts/train.py','--archive',archive,'--size',size,'--seed',seed,'--loss',loss)

## Repeated exploratory evaluation — only after all 18 new checkpoints exist

This previously examined 1,004-image set is NOT a fresh blind test.

In [ ]:
for size in (140,280,420):
    for seed in (20261028,20261029,20261030):
        for loss in ('ce','poly1'):
            out=base/f'results/eval_size{size}_seed{seed}_{loss}.json'
            if not out.exists():
                cmd('scripts/evaluate.py','--archive',archive,'--size',size,'--seed',seed,'--loss',loss)
cmd('scripts/report.py')
print((base/'results/REPORT_EN.md').read_text())

In [ ]:
# Export results only. Existing original archive and 70 source images are NOT duplicated.
import shutil
export=shutil.make_archive('/content/Nazih_Training_Set_Size_Results','zip',root_dir=base,base_dir='results')
print(export)
files.download(export)